---
## Warframes

In [ ]:
import json
import pydantic
from typing import List, Optional
from pydantic import BaseModel, Field

warframes = json.loads(open("../data/warframes.json").read())


In [ ]:
class Ability(BaseModel):
    """Represents a Warframe ability."""

    ability_unique_name: str = Field(..., alias="abilityUniqueName")
    ability_name: str = Field(..., alias="abilityName")
    description: str


class Warframe(BaseModel):
    """Represents a Warframe character and its base statistics."""

    unique_name: str = Field(..., alias="uniqueName")
    name: str
    parent_name: str = Field(..., alias="parentName")
    description: str
    health: int
    shield: int
    armor: int
    stamina: int
    power: int
    codex_secret: bool = Field(..., alias="codexSecret")
    mastery_req: int = Field(..., alias="masteryReq")
    sprint_speed: float = Field(..., alias="sprintSpeed")
    abilities: List[Ability]
    product_category: str = Field(..., alias="productCategory")
    passive_description: Optional[str] = Field(None, alias="passiveDescription")
    exalted: Optional[List[str]] = None
    long_description: Optional[str] = Field(None, alias="longDescription")

warframes = [Warframe(**wf) for wf in warframes]

In [ ]:
gyre_prime = next(wf for wf in warframes if (("Gyre" in wf.name) and ("Prime" in wf.name))).model_dump()
gyre_prime

---
## Recipes

In [ ]:
recipes = json.loads(open("../data/recipes.json").read())

In [ ]:
class Ingredient(BaseModel):
    """Represents a component required to craft an item."""

    item_type: str = Field(..., alias="ItemType")
    item_count: int = Field(..., alias="ItemCount")
    product_category: str = Field(..., alias="ProductCategory")


class SecretIngredient(BaseModel):
    """Represents a hidden component required to craft an item."""

    item_type: str = Field(..., alias="ItemType")
    item_count: int = Field(..., alias="ItemCount")


class Blueprint(BaseModel):
    """Represents a Warframe crafting recipe and its requirements."""

    unique_name: str = Field(..., alias="uniqueName")
    result_type: str = Field(..., alias="resultType")
    build_price: int = Field(..., alias="buildPrice")
    build_time: int = Field(..., alias="buildTime")
    skip_build_time_price: int = Field(..., alias="skipBuildTimePrice")
    consume_on_use: bool = Field(..., alias="consumeOnUse")
    num: int
    codex_secret: bool = Field(..., alias="codexSecret")
    ingredients: List[Ingredient]
    secret_ingredients: List[SecretIngredient] = Field(..., alias="secretIngredients")
    exclude_from_codex: Optional[bool] = Field(None, alias="excludeFromCodex")
    prime_selling_price: Optional[int] = Field(None, alias="primeSellingPrice")
    always_available: Optional[bool] = Field(None, alias="alwaysAvailable")

recipes = [Blueprint(**recipe) for recipe in recipes]

In [ ]:
next(r for r in recipes if r.result_type == gyre_prime["unique_name"]).model_dump()

---
## Resources

In [ ]:
resources = json.loads(open("../data/resources.json").read())


In [ ]:
class ResourceItem(BaseModel):
    unique_name: str = Field(alias="uniqueName")
    name: str
    description: str
    codex_secret: bool = Field(alias="codexSecret")
    parent_name: str = Field(alias="parentName")
    exclude_from_codex: Optional[bool] = Field(None, alias="excludeFromCodex")
    show_in_inventory: Optional[bool] = Field(None, alias="showInInventory")
    long_description: Optional[str] = Field(None, alias="longDescription")
    prime_selling_price: Optional[int] = Field(None, alias="primeSellingPrice")

In [ ]:
resources = [ResourceItem(**r) for r in resources]

In [ ]:
gyre_prime_blueprint = next(r for r in recipes if r.result_type == gyre_prime['unique_name'])

total_seconds = gyre_prime_blueprint.build_time
days = total_seconds // (24 * 3600)
hours = (total_seconds % (24 * 3600)) // 3600
minutes = (total_seconds % 3600) // 60

print(f"--- {gyre_prime['name']} ---")
print(f"Description: {gyre_prime['description']}")
print(f"\nBuild Time: {int(days)}d {int(hours)}h {int(minutes)}m")
print("\nRequired Resources:")
for ing in gyre_prime_blueprint.ingredients:
    res_name = "Unknown"
    for res in resources:
        if res.unique_name == ing.item_type:
            res_name = res.name
            break
    print(f"- {ing.item_count}x {res_name} ({ing.item_type})")